In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


spark = SparkSession.builder \
    .appName("ExpenseAnalysis") \
    .getOrCreate()


data = [
    ("2026-05-01", "Food", 250.0, 1),
    ("2026-05-02", "Transport", 100.0, 1),
    ("2026-05-03", "Shopping", 2000.0, 1),
    ("2026-05-04", "Food", 300.0, 2),
    ("2026-05-05", "Bills", 1500.0, 2),
    ("2026-05-06", "Entertainment", 4500.0, 1)
]


columns = ["date", "category", "amount", "user_id"]


df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()


df = df.withColumn(
    "expense_date",
    to_date(col("date"))
)


df = df.withColumn(
    "month",
    date_format(col("expense_date"), "yyyy-MM")
)

print("Data With Month Column:")
df.show()


monthly_spend = df.groupBy(
    "user_id",
    "month"
).agg(
    sum("amount").alias("total_monthly_spend")
)

print("Monthly Spend Per User:")
monthly_spend.show()


average_amount = df.agg(
    avg("amount").alias("average")
).collect()[0]["average"]

print("Average Expense Amount:", average_amount)


anomaly_df = df.filter(
    col("amount") > average_amount * 2
)

print("Potential Unusual Spending:")
anomaly_df.show()


spark.stop()

Original Data:
+----------+-------------+------+-------+
|      date|     category|amount|user_id|
+----------+-------------+------+-------+
|2026-05-01|         Food| 250.0|      1|
|2026-05-02|    Transport| 100.0|      1|
|2026-05-03|     Shopping|2000.0|      1|
|2026-05-04|         Food| 300.0|      2|
|2026-05-05|        Bills|1500.0|      2|
|2026-05-06|Entertainment|4500.0|      1|
+----------+-------------+------+-------+

Data With Month Column:
+----------+-------------+------+-------+------------+-------+
|      date|     category|amount|user_id|expense_date|  month|
+----------+-------------+------+-------+------------+-------+
|2026-05-01|         Food| 250.0|      1|  2026-05-01|2026-05|
|2026-05-02|    Transport| 100.0|      1|  2026-05-02|2026-05|
|2026-05-03|     Shopping|2000.0|      1|  2026-05-03|2026-05|
|2026-05-04|         Food| 300.0|      2|  2026-05-04|2026-05|
|2026-05-05|        Bills|1500.0|      2|  2026-05-05|2026-05|
|2026-05-06|Entertainment|4500.0|   